# Data Generation and Preparation

A. YouTube → Whisper → HF Dataset

OR

B. Upload Recordings → Whisper → HF Dataset

Pipeline:

1. Enter YouTube URLs OR upload .mp3 or .wav files.
2. Whisper transcribes the audio → a single JSON file you can edit.
3. Audio segments are concatenated and then split into sentences using nltk.
4. Sentences are combined into ≤ 30-second clips, one row per clip (audio, text, source).
5. The resulting datasets.Dataset is pushed to Hugging Face under the org/repo of your choice.

In [9]:
# ▸ Installs (quiet)
!pip -q install --upgrade ffmpeg-python \
    git+https://github.com/openai/whisper.git \
    soundfile datasets

# and if you are downloading from YouTube
!pip -q install --upgrade yt_dlp

# ▸ Imports
import os
import json
import subprocess
import uuid
import torch
import whisper
import glob

from pathlib import Path

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# 1. Import libraries and set up model and audio directory
import torch
import os
from pathlib import Path

# Whisper model & hardware configuration
MODEL_SIZE = "turbo"  # tiny | base | small | medium | large-v3 | turbo
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Directory for all audio files
AUDIO_DIR = Path("/content/audio")
AUDIO_DIR.mkdir(exist_ok=True, parents=True)

print(f"Using {MODEL_SIZE} model on {DEVICE}")


Using turbo model on cuda


In [17]:
# 2. Upload audio files (.mp3, .wav, .m4a) from local drive to Colab
from google.colab import files

uploaded = files.upload()  # dialog to pick files
for name, _ in uploaded.items():
    dest = AUDIO_DIR / name  # move file into AUDIO_DIR
    os.rename(name, dest)
    print("📁 Saved", dest)
print(f"✅ All uploads saved to {AUDIO_DIR}")


Saving Pixel 10Pro Review Good News and Bad News!_MP3.mp3 to Pixel 10Pro Review Good News and Bad News!_MP3.mp3
Saving Smartphone Awards 2025!_MP3.mp3 to Smartphone Awards 2025!_MP3.mp3
📁 Saved /content/audio/Pixel 10Pro Review Good News and Bad News!_MP3.mp3
📁 Saved /content/audio/Smartphone Awards 2025!_MP3.mp3
✅ All uploads saved to /content/audio


# Downloading the Audio forn Youtube


In [12]:
# # Add / paste any number of YouTube links here
# YOUTUBE_URLS = [
#     "https://www.youtube.com/watch?v=9GSDvO0LFFE",
#     "https://www.youtube.com/watch?v=Hc0aqOEU2w8",
#     "https://www.youtube.com/watch?v=kBX5WH9b4M4",
#     "https://www.youtube.com/watch?v=DqAKQwagCDg",
#     "https://www.youtube.com/watch?v=Tl8RS0sR-qA",
#     "https://www.youtube.com/watch?v=i63u-iAnhuk",
#     "https://www.youtube.com/watch?v=eLusurjBcCs",
# ]

# # --- OPTIONAL: Provide YouTube cookies for authentication ---
# # If you encounter 'Sign in to confirm you're not a bot' errors,
# # export your YouTube cookies (e.g., using a browser extension like 'EditThisCookie')
# # and save them to a file in Colab (e.g., /content/youtube_cookies.txt).
# # Then uncomment the line below and set the path to your cookies file.
# COOKIES_FILE = Path('/content/local_youtube_cookies.txt')
# # COOKIES_FILE = None # Set to None if not using cookies

# for url in YOUTUBE_URLS:

#     # Prepare common yt-dlp arguments
#     yt_dlp_args = [
#         "yt-dlp",
#         "-q",
#         "-x",
#         "--audio-format", "m4a",
#         "-o", str(AUDIO_DIR / "%(title)s.%(ext)s"),
#     ]
#     if COOKIES_FILE and COOKIES_FILE.exists():
#         yt_dlp_args.extend(["--cookies", str(COOKIES_FILE)])

#     expected_filename = None

#     try:
#         # Attempt to get the title using yt-dlp's --print option
#         # Use a subset of yt_dlp_args for title extraction, removing format/output args
#         title_args = ["yt-dlp"]
#         if COOKIES_FILE and COOKIES_FILE.exists():
#             title_args.extend(["--cookies", str(COOKIES_FILE)])
#         title_args.extend(["--print", "%(title)s", url])

#         process = subprocess.run(
#             title_args,
#             capture_output=True,
#             text=True,
#             check=True,
#         )

#         title = process.stdout.strip()
#         expected_filename = AUDIO_DIR / f"{title}.m4a"

#     except Exception as e:
#         print(
#             f"Warning: Could not determine expected filename for {url}: {e}"
#         )
#         if hasattr(e, 'stderr'):
#             print(f"yt-dlp stderr: {e.stderr}")

#         # Fallback or skip if filename cannot be determined
#         print(
#             f"⬇️ Downloading {url} (without checking for existing file)"
#         )

#         try:
#             subprocess.run(
#                 yt_dlp_args + [url],
#                 check=True,
#                 capture_output=True, # Capture stderr for debugging
#                 text=True
#             )
#         except subprocess.CalledProcessError as sub_e:
#             print(f"Error during direct download for {url}: {sub_e}")
#             print(f"yt-dlp stderr during direct download: {sub_e.stderr}")

#         continue  # Move to the next URL

#     if expected_filename and expected_filename.exists():
#         print(
#             f"⏭️ Skipping download for {url} "
#             f"(already exists as {expected_filename.name})"
#         )

#     else:
#         print(f"⬇️ Downloading {url}")

#         # Use video title as file name, keep original audio codec when possible
#         try:
#             subprocess.run(
#                 yt_dlp_args + [url],
#                 check=True,
#                 capture_output=True, # Capture stderr for debugging
#                 text=True
#             )
#         except subprocess.CalledProcessError as sub_e:
#             print(f"Error during download for {url}: {sub_e}")
#             print(f"yt-dlp stderr during download: {sub_e.stderr}")

# print("✅ All YouTube audio saved to", AUDIO_DIR)


yt-dlp stderr: WARNING: [youtube] The provided YouTube account cookies are no longer valid. They have likely been rotated in the browser as a security measure. For tips on how to effectively export YouTube cookies, refer to  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies .
ERROR: [youtube] 9GSDvO0LFFE: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

⬇️ Downloading https://www.youtube.com/watch?v=9GSDvO0LFFE (without checking for existing file)
Error during direct download for https://www.youtube.com/watch?v=9GSDvO0LFFE: Command '['yt-dlp', '-q', '-x', '--audio-format', 'm4a', '-o', '/content/audio/%(title)s.%(ext)s', '--cookies', '/content/local_youtube_cookie

In [56]:
# # Create a local cookies file with the provided Netscape format content
# local_cookies_file_path = Path('/content/local_youtube_cookies.txt')

# netflix_cookies_content = """# Netscape HTTP Cookie File
# # http://curl.haxx.se/rfc/cookie_spec.html
# # This file was generated by Cookie-Editor
# #HttpOnly_.youtube.com	TRUE	/	TRUE	1814243069	__Secure-3PSID	g.a000-QiDj28RTsP01cJ1UyEDbaJATzEfPYOJnL6J4Lu4bDuBMrbjkMYXVgSfLnwh42sj9xXFvgACgYKAaQSARASFQHGX2MiJYSRHNVMql1Q2G4LOGuG7xoVAUF8yKqpaViFnubjZc6PU_zYHFRY0076
# #HttpOnly_.youtube.com	TRUE	/	TRUE	1811576194	__Secure-1PSIDTS	sidts-CjUBhkeRdwypGiYi33hCH_QHH8qlPOwS3bIGSQNETOM2jc1rvVqjtoVqnSSvku4T8kOc0-O96BAA
# .youtube.com	TRUE	/	TRUE	1814243069	__Secure-3PAPISID	vRT9YaTQJ022sys-/Al89JzPf7NLe9fd4I
# #HttpOnly_.youtube.com	TRUE	/	TRUE	1811576194	__Secure-3PSIDCC	AKEyXzWUW6VEwK9ktVdXvUxvW3p-xE2Jg-Ag26U-RccsKJecTezHqUSJ8m3mvGc5tYjIJE5fC0c
# #HttpOnly_.youtube.com	TRUE	/	TRUE	1811576194	__Secure-3PSIDTS	sidts-CjUBhkeRdwypGiYi33hCH_QHH8qlPOwS3bIGSQNETOM2jc1rvVqjtoVqnSSvku4T8kOc0-O96BAA
# #HttpOnly_.youtube.com	TRUE	/	TRUE	1791178149	__Secure-BUCKET	CPMG
# #HttpOnly_.youtube.com	TRUE	/	TRUE	1814243069	LOGIN_INFO	AFmmF2swRgIhAKiZtaRIXhB976JwTDdMSFe2UOP0r1V2SnHS1Z0E_bjWAiEA5LMVIeeOHPGivIEIssaX855PQ3sUf_5Rm59kb7wYv8Q:QUQ3MjNmelF2d1VYc1VGNDV3RnVDaGNEdFRrS1AwZ2ZELWZSS2tzYUkxbjYtODVfUi1lVUtqcVJpOW1KanF6dzR1OXo2bXFmeUxuLVpyQlNXekl3TmczdzhPT2Q3eVZOVEt1RHB1ZzlOVm51U2FuaUNDc3hiYVJQcG1VNmU0cnpURXh5aC1IV1ZSXzc0SDhNVEF1YjJBVzZ5ZFZHZm9ONUVR
# .youtube.com	TRUE	/	TRUE	1814599530	PREF	f4=4000000&tz=Asia.Calcutta&f7=18100&f6=40000000
# """

# with open(local_cookies_file_path, 'w') as f:
#     f.write(netflix_cookies_content)

# print(f"Local cookies file created at: {local_cookies_file_path}")

In [18]:
# 4. Compute durations and collect audio file paths
import subprocess, json
from itertools import chain

def get_duration_sec(path: Path) -> float:
    """Return the length of the audio file in seconds using ffprobe."""
    pr = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "json", str(path)],
        capture_output=True, text=True
    )
    try:
        return float(json.loads(pr.stdout)["format"]["duration"])
    except (json.JSONDecodeError, KeyError, ValueError) as e:
        print(f"⚠️ Error getting duration for {path}: {e}")
        print(f"ffprobe stdout: {pr.stdout}")
        print(f"ffprobe stderr: {pr.stderr}")
        return 0.0

# Collect all audio files from AUDIO_DIR
audio_paths = sorted(chain(
    AUDIO_DIR.glob("*.mp3"),
    AUDIO_DIR.glob("*.wav"),
    AUDIO_DIR.glob("*.m4a"),
    AUDIO_DIR.glob("*.webm"),  # include WebM if present
))
print(f"👀 Found {len(audio_paths)} audio files")

total_sec = 0
# 5. Print duration of each file and total duration
total_sec = 0
for p in audio_paths:
    dur_sec = get_duration_sec(p)
    total_sec += dur_sec
    print(f"🎵 {p.name:<40} {dur_sec/60:6.2f} min")

print(f"\n🕒 Total duration: {total_sec/60:0.2f} min ({total_sec/3600:0.2f} h)")



👀 Found 5 audio files
🎵 My FBI Declassified Story_MP3.mp3          9.43 min
🎵 Pixel 10Pro Review Good News and Bad News!_MP3.mp3  20.32 min
🎵 Smartphone Awards 2025!_MP3.mp3           32.63 min
🎵 Switching from iPhone to Android_MP3.mp3   9.12 min
🎵 The Honey Scam Explained_MP3.mp3          10.87 min

🕒 Total duration: 82.37 min (1.37 h)


In [19]:
# ── clean + normalise audio to a new folder ───────────────────────────

from pathlib import Path
import subprocess
import json
from IPython.display import Audio, display

CLEANED_AUDIO_DIR = Path("/content/audio_clean")
CLEANED_AUDIO_DIR.mkdir(exist_ok=True)

# ── collect audio files ───────────────────────────────────────────────
# Use audio_paths from the previous cell that lists the original audio files
# audio_paths is already defined in the cell above this one.

def clean_one(src: Path) -> Path:
    """
    Applies a series of FFmpeg filters to clean and normalize audio.

    Args:
        src: Path to the source audio file.

    Returns:
        Path to the cleaned output audio file in CLEANED_AUDIO_DIR.
    """

    out = CLEANED_AUDIO_DIR / f"{src.stem}_clean.wav"

    filt = (
        "highpass=f=80,"          # Apply a high-pass filter to remove low-frequency noise below 80Hz
        "afftdn,"                # Apply an FFT-based denoiser to reduce broadband noise
        "loudnorm=I=-16:LRA=11:TP=-1.5,"  # Apply loudness normalization
        "dynaudnorm=f=200,"      # Apply dynamic audio normalization
        "apad=pad_dur=0.1"       # Add 100 ms silence padding at end
    )

    subprocess.run(
        [
            "ffmpeg",
            "-loglevel",
            "error",
            "-y",
            "-i",
            str(src),
            "-af",
            filt,
            "-ar",
            "24000",
            "-ac",
            "1",  # Set audio sample rate to 24000 Hz and output as mono
            str(out),
        ],
        check=True,
    )

    return out

In [ ]:
# # 7. Extract and clean first audio file for demonstration
# first_original_audio_path = audio_paths[0]
# cleaned_path = clean_one(first_original_audio_path)
# print(f"Original file: {first_original_audio_path}")
# print(f"Cleaned file:  {cleaned_path}")
# # Optionally, play audio in notebook:
# # display(Audio(str(cleaned_path), autoplay=True))


In [20]:
# Extract and clean the first 10 seconds of the first audio file for comparison

# Get the first audio file path from the original list
first_original_audio_path = audio_paths[0]

print("Original audio (first 10 seconds):")

temp_original_clip_path = Path("/content/temp_original_clip_10s.wav")

subprocess.run([
    "ffmpeg",
    "-loglevel", "error",
    "-y",
    "-i", str(first_original_audio_path),
    "-ss", "0",
    "-t", "10",     # first 10 seconds
    str(temp_original_clip_path)
], check=True)

display(Audio(str(temp_original_clip_path)))

original_clip_duration = get_duration_sec(temp_original_clip_path)

print(f"Duration: {original_clip_duration:.3f} seconds")


print("\nCleaning the first 10 seconds...")

# clean_one expects a Path object
temp_cleaned_clip_path = clean_one(temp_original_clip_path)

print("\nCleaned audio (first 10 seconds):")

display(Audio(str(temp_cleaned_clip_path)))

cleaned_clip_duration = get_duration_sec(temp_cleaned_clip_path)

print(f"Duration: {cleaned_clip_duration:.3f} seconds")

Original audio (first 10 seconds):


Duration: 10.000 seconds

Cleaning the first 10 seconds...

Cleaned audio (first 10 seconds):


Duration: 10.070 seconds


# Clean All Audio Files

Apply the cleaning pipeline to every audio file.

In [21]:
cleaned_paths = []
total_clean_sec = 0

for p in audio_paths:

    cleaned_file = CLEANED_AUDIO_DIR / f"{p.stem}_clean.wav"

    if cleaned_file.exists():

        print(f"⏭️ Skipping cleaning for {p.name} (already exists)")

        cleaned_paths.append(cleaned_file)

        dur = get_duration_sec(cleaned_file)
        total_clean_sec += dur

    else:

        print(f"🧹 Cleaning {p.name} ...", end="")

        cleaned = clean_one(p)

        cleaned_paths.append(cleaned)

        dur = get_duration_sec(cleaned)

        total_clean_sec += dur

        print(f" done ({dur/60:.2f} min)")


print(f"\n✅ All cleaned files in {CLEANED_AUDIO_DIR}")

print(
    f"📦 Total newly cleaned duration: "
    f"{total_clean_sec/60:.2f} min "
    f"({total_clean_sec/3600:.2f} h)"
)

🧹 Cleaning My FBI Declassified Story_MP3.mp3 ... done (9.43 min)
🧹 Cleaning Pixel 10Pro Review Good News and Bad News!_MP3.mp3 ... done (20.32 min)
🧹 Cleaning Smartphone Awards 2025!_MP3.mp3 ... done (32.63 min)
🧹 Cleaning Switching from iPhone to Android_MP3.mp3 ... done (9.12 min)
🧹 Cleaning The Honey Scam Explained_MP3.mp3 ... done (10.87 min)

✅ All cleaned files in /content/audio_clean
📦 Total newly cleaned duration: 82.37 min (1.37 h)


# Verify Cleaned Audio

In [22]:
# Regenerate cleaned_paths

cleaned_paths = sorted(
    CLEANED_AUDIO_DIR.glob("*.wav")
)

print(f"🔎 Found {len(cleaned_paths)} cleaned audio files")

total_sec = 0

for p in cleaned_paths:

    dur_sec = get_duration_sec(p)

    total_sec += dur_sec

    print(
        f" • {p.name:<40} "
        f"{dur_sec/60:6.2f} min"
    )

print(
    f"\n📦 Total duration: "
    f"{total_sec/60:.2f} min "
    f"({total_sec/3600:.2f} h)"
)

🔎 Found 6 cleaned audio files
 • My FBI Declassified Story_MP3_clean.wav    9.43 min
 • Pixel 10Pro Review Good News and Bad News!_MP3_clean.wav  20.32 min
 • Smartphone Awards 2025!_MP3_clean.wav     32.63 min
 • Switching from iPhone to Android_MP3_clean.wav   9.12 min
 • The Honey Scam Explained_MP3_clean.wav    10.87 min
 • temp_original_clip_10s_clean.wav           0.17 min

📦 Total duration: 82.54 min (1.38 h)


## Transcribe Audio with Whisper
Generate word-level transcripts for every cleaned audio file.
it is gonna use neural net for sentence detection

In [23]:
 import whisper

model = whisper.load_model(MODEL_SIZE, device=DEVICE)

TRANSCRIPTS_DIR = Path("/content/transcripts")
TRANSCRIPTS_DIR.mkdir(exist_ok=True)

for audio_path in cleaned_paths:

    fname = audio_path.stem  # still a Path – fine
    out_json = TRANSCRIPTS_DIR / f"{fname}_whisper.json"

    if out_json.exists():
        print(
            f"⏭️ Skipping transcription for {fname} (already exists)"
        )
        continue

    print(f"\n🎤 Transcribing {fname} ...")

    result = model.transcribe(
        str(audio_path),      # convert Path → str
        word_timestamps=True,
        fp16=(DEVICE == "cuda"),
        verbose=False,
    )

    with out_json.open("w") as f:
        json.dump(result, f, indent=2)

    print("✅ Saved →", out_json)

print("\n🏁 Done! All transcripts in", TRANSCRIPTS_DIR)

100%|█████████████████████████████████████| 1.51G/1.51G [00:18<00:00, 87.7MiB/s]



🎤 Transcribing My FBI Declassified Story_MP3_clean ...
Detected language: English


100%|██████████| 56581/56581 [00:40<00:00, 1405.92frames/s]


✅ Saved → /content/transcripts/My FBI Declassified Story_MP3_clean_whisper.json

🎤 Transcribing Pixel 10Pro Review Good News and Bad News!_MP3_clean ...
Detected language: English


100%|██████████| 121937/121937 [01:12<00:00, 1684.30frames/s]


...

✅ Saved → /content/transcripts/Switching from iPhone to Android_MP3_clean_whisper.json

🎤 Transcribing The Honey Scam Explained_MP3_clean ...
Detected language: English


100%|██████████| 65215/65215 [00:37<00:00, 1716.97frames/s]


✅ Saved → /content/transcripts/The Honey Scam Explained_MP3_clean_whisper.json

🎤 Transcribing temp_original_clip_10s_clean ...
Detected language: English


100%|██████████| 1007/1007 [00:00<00:00, 1165.67frames/s]

✅ Saved → /content/transcripts/temp_original_clip_10s_clean_whisper.json

🏁 Done! All transcripts in /content/transcripts


In [26]:
import shutil

shutil.make_archive(
    "/content/transcripts",
    "zip",
    "/content/transcripts"
)

'/content/transcripts.zip'

In [27]:
from google.colab import files

files.download("/content/transcripts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Login to Hugging Face

In [29]:
from huggingface_hub import whoami

try:
    print(whoami())
except Exception as e:
    print("Not logged in")
    print(e)

{'type': 'user', 'id': '6863e13ee53f79ac462b4e1b', 'name': 'Mrinal007', 'fullname': 'Mrinal seth', 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/7cVxehNtnBpGjqG4hUFpK.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'colab_dataset_upload', 'role': 'fineGrained', 'createdAt': '2026-05-29T08:42:32.656Z', 'fineGrained': {'canReadGatedRepos': False, 'global': [], 'scoped': [{'entity': {'_id': '6863e13ee53f79ac462b4e1b', 'type': 'user', 'name': 'Mrinal007'}, 'permissions': ['repo.access.read', 'repo.content.read', 'repo.write']}]}}}}


In [28]:
from huggingface_hub import login, HfApi
from huggingface_hub.utils import LocalTokenNotFoundError

try:

    HfApi().whoami()

    print("Already logged into Hugging Face.")

except LocalTokenNotFoundError:

    print(
        "Not logged into Hugging Face. "
        "Logging in..."
    )

    login()

Not logged into Hugging Face. Logging in...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Chunk Audio Respecting Sentence Boundaries

Prepare clips suitable for CSM-1B / Orpheus training.



In [30]:
from huggingface_hub import create_repo

create_repo(
    repo_id="Mrinal007/mkbhd_tts_medium_clean",
    repo_type="dataset",
    exist_ok=True
)

RepoUrl('https://huggingface.co/datasets/Mrinal007/mkbhd_tts_medium_clean', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Mrinal007/mkbhd_tts_medium_clean')

In [31]:
from pathlib import Path
import json
import subprocess
import uuid
import datasets
import os

# ============================================================
# Hugging Face Dataset Configuration
# ============================================================

HF_ORG = "Mrinal007"
REPO_NAME = "mkbhd_tts_medium_clean"

# Maximum audio length (seconds)
MAX_SEC = 30

# ============================================================
# Audio Configuration (Required for CSM-1B)
# ============================================================

sampling_rate = 24000

# ============================================================
# Full Hugging Face Repository Path
# ============================================================

HF_DATASET_REPO = f"{HF_ORG}/{REPO_NAME}"

print(f"Dataset will be uploaded to: {HF_DATASET_REPO}")

Dataset will be uploaded to: Mrinal007/mkbhd_tts_medium_clean


In [32]:
# Chunk every audio-transcript pair ≤ MAX_SEC

from pathlib import Path
import json, subprocess, uuid, os, datasets

# ── dirs ─────────────────────────────────────────────
TRANSCRIPTS_DIR = Path("/content/transcripts")
CLIPS_ROOT      = Path("/content/clips")
CLIPS_ROOT.mkdir(exist_ok=True)

CLEANED_AUDIO_DIR = Path("/content/audio_clean")

rows = []  # final HuggingFace rows


def chunk_one(audio_path: Path, json_path: Path):
    """Return list[dict] -> {audio, text, source} for one file."""

    with open(json_path) as f:
        data = json.load(f)

    segments = data["segments"]

    # ----- build word-level metadata -----------------
    word_meta, full_text, char_pos = [], [], 0

    for seg in segments:
        for w in seg["words"]:
            tok = w["word"].strip()

            if not tok:
                continue

            start_char = char_pos
            end_char = char_pos + len(tok)

            word_meta.append({
                "char0": start_char,
                "char1": end_char,
                "t0": w["start"],
                "t1": w["end"]
            })

            full_text.extend([tok, " "])
            char_pos = end_char + 1

    full_text = "".join(full_text).rstrip()

    # ----- sentence spans with NLTK ------------------
    import nltk, itertools
    from nltk.tokenize import PunktSentenceTokenizer

    nltk.download("punkt", quiet=True)

    tokenizer = PunktSentenceTokenizer()
    sent_spans = list(tokenizer.span_tokenize(full_text))

    sentences, w_idx = [], 0

    for c0, c1 in sent_spans:

        # map chars -> word indices -> times
        first_w = w_idx

        while (
            first_w < len(word_meta)
            and word_meta[first_w]["char1"] <= c0
        ):
            first_w += 1

        last_w = first_w

        while (
            last_w < len(word_meta)
            and word_meta[last_w]["char0"] < c1
        ):
            last_w += 1

        if first_w == last_w:
            continue

        s_time = word_meta[first_w]["t0"]
        e_time = word_meta[last_w - 1]["t1"]

        text = full_text[c0:c1]

        sentences.append({
            "start": s_time,
            "end": e_time,
            "text": text
        })

        w_idx = last_w

    # ----- bundle sentences ≤ MAX_SEC and write clips -----
    clip_rows, bundle = [], None

    def flush(b):
        if b is None:
            return

        st, et, tx, out_dir = b

        out_dir.mkdir(exist_ok=True, parents=True)

        clip_path = out_dir / f"clip_{uuid.uuid4().hex}.wav"

        subprocess.run(
            [
                "ffmpeg",
                "-loglevel", "error",
                "-y",
                "-i", str(audio_path),
                "-ss", f"{st}",
                "-to", f"{et}",
                "-ar", str(sampling_rate),
                "-ac", "1",
                str(clip_path),
            ],
            check=True,
        )

        clip_rows.append(
            {
                "audio": str(clip_path),
                "text": tx.strip(),
                "source": "0",
            }
        )

    # iterate sentences
    out_dir = CLIPS_ROOT / audio_path.stem

    for s in sentences:

        st, et, tx = s["start"], s["end"], s["text"]

        dur = et - st

        # skip single sentences that are too long
        if dur > MAX_SEC:
            continue

        if bundle is None:
            bundle = [st, et, tx, out_dir]
            continue

        b_st, b_et, b_tx, _ = bundle

        if (et - b_st) <= MAX_SEC:
            bundle = [b_st, et, b_tx + " " + tx, out_dir]
        else:
            flush(bundle)
            bundle = [st, et, tx, out_dir]

    flush(bundle)  # final bundle

    return clip_rows


for audio_path in cleaned_paths:

    stem = audio_path.stem
    json_path = TRANSCRIPTS_DIR / f"{stem}_whisper.json"

    if not json_path.exists():
        print(f"⚠️ {stem} → skipping (no transcript)")
        continue

    print(f"\n🔨 Chunking {stem} ...")

    clip_rows = chunk_one(audio_path, json_path)

    rows.extend(clip_rows)

    print(f"   ↳ {len(clip_rows)} clips")

print(
    f"\n🎉 Created {len(rows)} total clips across "
    f"{len(cleaned_paths)} source files."
)


🔨 Chunking My FBI Declassified Story_MP3_clean ...
   ↳ 21 clips

🔨 Chunking Pixel 10Pro Review Good News and Bad News!_MP3_clean ...
   ↳ 46 clips

🔨 Chunking Smartphone Awards 2025!_MP3_clean ...
   ↳ 71 clips

🔨 Chunking Switching from iPhone to Android_MP3_clean ...
   ↳ 21 clips

🔨 Chunking The Honey Scam Explained_MP3_clean ...
   ↳ 25 clips

🔨 Chunking temp_original_clip_10s_clean ...
   ↳ 1 clips

🎉 Created 185 total clips across 6 source files.


In [33]:
print(cleaned_paths)

[PosixPath('/content/audio_clean/My FBI Declassified Story_MP3_clean.wav'), PosixPath('/content/audio_clean/Pixel 10Pro Review Good News and Bad News!_MP3_clean.wav'), PosixPath('/content/audio_clean/Smartphone Awards 2025!_MP3_clean.wav'), PosixPath('/content/audio_clean/Switching from iPhone to Android_MP3_clean.wav'), PosixPath('/content/audio_clean/The Honey Scam Explained_MP3_clean.wav'), PosixPath('/content/audio_clean/temp_original_clip_10s_clean.wav')]


In [34]:
print(rows)

[{'audio': '/content/clips/My FBI Declassified Story_MP3_clean/clip_1140900903b644e9bc047c9153ad7600.wav', 'text': "So, I've made a few references over the past couple years to that time that the FBI visited the studio, and I was asked not to talk about that for what I feel like are pretty obvious reasons. So for years, I haven't, but it turns out I am now finally able to tell that story. So, here we are. So for those of you who don't know, we have a studio here, and it's in a larger building with more tenants in it, obviously.", 'source': '0'}, {'audio': '/content/clips/My FBI Declassified Story_MP3_clean/clip_32402d22d2b0402ea80235b10cf6df8d.wav', 'text': "It's like an office building. It's not exactly like an office building, but, you know, there's a front desk and security and all that, and I've been in this building long enough that we have a pretty good relationship with them. So, we have this one rule, basically. So there's the guy at security at the front desk.", 'source': '0'}

In [37]:
print(
    f"Sample row text:\n"
    f"{rows[0]['text']}"
)

Sample row text:
So, I've made a few references over the past couple years to that time that the FBI visited the studio, and I was asked not to talk about that for what I feel like are pretty obvious reasons. So for years, I haven't, but it turns out I am now finally able to tell that story. So, here we are. So for those of you who don't know, we have a studio here, and it's in a larger building with more tenants in it, obviously.


In [38]:
# Push as HF dataset

ds = datasets.Dataset.from_list(rows)

# Set the column feature as "audio" so HF displays it nicely
# and uses streaming.

ds = ds.cast_column(
    "audio",
    datasets.Audio(sampling_rate=sampling_rate)
)

repo_id = f"{HF_ORG}/{REPO_NAME}"

if True:
    print(f"Pushing to {repo_id} ...")
    ds.push_to_hub(repo_id, private=False)
    print("✅ Done!")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Pushing to Mrinal007/mkbhd_tts_medium_clean ...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/185 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpc8l60gqy.parquet    :   0%|          |  584kB /  233MB            

✅ Done!
